In [1]:
from google.colab import drive
drive.mount('/content/drive/')

import os


Mounted at /content/drive/


In [2]:
# Finding the Data Directory
data_dir = '/content/drive/My Drive/Ml_IR'
os.chdir(data_dir)

# List the contents of the directory
print(os.listdir())

print("------------")


['IR_NN.ipynb', 'ir_spectroscopy_dataset.pt']
------------


In [90]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

data = torch.load('ir_spectroscopy_dataset.pt')
X_values = data['X_values']
Y_values = data['Y_values']
Labels = data['Labels']

#print(X_values)


def preprocess_data(X_values, Y_values, Labels, cut_front=10, cut_end=10):
    """
    Preprocess the X_values and Y_values tensors by trimming a few points from the beginning and end,
    and removing data points with all NaN values or only NaN intensities.

    Parameters:
    - X_values: torch.Tensor
        The tensor containing the X values (wavenumbers) of the IR spectra
    - Y_values: torch.Tensor
        The tensor containing the Y values of the IR spectra
    - Labels: torch.Tensor
        The tensor containing the labels corresponding to Y_values
    - cut_front: int, default=10
        The number of data points to cut from the front of each spectrum
    - cut_end: int, default=10
        The number of data points to cut from the end of each spectrum

    Returns:
    - processed_X: torch.Tensor
        The processed X values tensor
    - processed_Y: torch.Tensor
        The processed Y values tensor
    - processed_Labels: torch.Tensor
        The processed labels tensor
    """
    # Trim the front and end of each spectrum
    processed_X = X_values[cut_front:-cut_end]
    processed_Y = Y_values[:, cut_front:-cut_end]

    # Create a mask for non-NaN rows and rows that are not all NaN
    mask = ~torch.isnan(processed_Y).all(dim=1) & ~torch.isnan(processed_Y).any(dim=1)

    # Apply the mask to Y_values and Labels
    processed_Y = processed_Y[mask]
    processed_Labels = Labels[mask]

    return processed_X, processed_Y, processed_Labels

# Example usage:
processed_X_values, processed_Y_values, processed_Labels = preprocess_data(X_values, Y_values, Labels)
print(f"Original X shape: {X_values.shape}, Processed X shape: {processed_X_values.shape}")
print(f"Original Y shape: {Y_values.shape}, Processed Y shape: {processed_Y_values.shape}")
print(f"Original labels: {Labels.shape}, Processed labels: {processed_Labels.shape}")

#----// NN //---------------------------


class NN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        #self.g = torch.Generator().manual_seed(2147483647)
        self.g = torch.Generator()

        self.W1 = torch.nn.Parameter(torch.randn((980, 4000), generator=self.g))
        self.b1 = torch.nn.Parameter(torch.randn(4000, generator=self.g))
        self.W2 = torch.nn.Parameter(torch.randn((4000, 2000), generator=self.g))
        self.b2 = torch.nn.Parameter(torch.randn(2000, generator=self.g))
        self.W3 = torch.nn.Parameter(torch.randn((2000, 500), generator=self.g))
        self.b3 = torch.nn.Parameter(torch.randn(500, generator=self.g))
        self.W4 = torch.nn.Parameter(torch.randn((500, 28), generator=self.g))
        self.b4 = torch.nn.Parameter(torch.randn(28, generator=self.g))

        print(f"The NN has: {sum(p.numel() for p in self.parameters())} parameters")

    def forward(self, Y_values, Labels):
        h = torch.tanh(Y_values @ self.W1 + self.b1)
        h2 = torch.tanh(h @ self.W2 + self.b2)
        h3 = torch.tanh(h2 @ self.W3 + self.b3)
        logits = h3 @ self.W4 + self.b4
        out = torch.sigmoid(logits)  # Use sigmoid to get values between 0 and 1

        # Create a target tensor from Labels
        target = Labels.to(logits.device)

        # Compute binary cross-entropy loss
        loss = torch.nn.functional.binary_cross_entropy(out, target)

        return loss, out

    def __call__(self, data_tensor):
        # Ensure data_tensor is on the same device as the model
        data_tensor = data_tensor.to(self.W1.device)
        
        h = torch.tanh(data_tensor @ self.W1 + self.b1)
        h2 = torch.tanh(h @ self.W2 + self.b2)
        h3 = torch.tanh(h2 @ self.W3 + self.b3)
        logits = h3 @ self.W4 + self.b4
        out = torch.sigmoid(logits)  # Use sigmoid to get values between 0 and 1

        return out

nn_model = NN()
optimizer = torch.optim.SGD(nn_model.parameters(), lr=0.1)


# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move the model to the GPU
nn_model = nn_model.to(device)

# Move optimizer to GPU
optimizer = torch.optim.SGD(nn_model.parameters(), lr=0.1)

# Move data tensors to GPU
processed_Y_values = processed_Y_values.to('cuda')
processed_Labels = processed_Labels.to('cuda')

print("Model and data moved to GPU successfully.")


Original X shape: torch.Size([1000]), Processed X shape: torch.Size([980])
Original Y shape: torch.Size([141, 1000]), Processed Y shape: torch.Size([139, 980])
Original labels: torch.Size([141, 28]), Processed labels: torch.Size([139, 28])
The NN has: 12940528 parameters
Using device: cuda
Model and data moved to GPU successfully.


<ipython-input-90-727536535c37>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load('ir_spectroscopy_dataset.pt')


In [91]:

#----// NN //---------------------------

optimizer = torch.optim.SGD(nn_model.parameters(), lr=3)

epoch = 0
while True:
    optimizer.zero_grad()
    loss, _ = nn_model.forward(processed_Y_values, processed_Labels)
    loss.backward()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")
    optimizer.step()

    if loss.item() < 1e-2:
        print("Reached Stop Point!!")
        print(f"Converged after {epoch+1} epochs.")
        break

    epoch += 1







Epoch 1, Loss: 24.405282974243164
Epoch 2, Loss: 21.468366622924805
Epoch 3, Loss: 21.68050765991211
Epoch 4, Loss: 21.508214950561523
Epoch 5, Loss: 18.948192596435547
Epoch 6, Loss: 16.68051528930664
Epoch 7, Loss: 15.370771408081055
Epoch 8, Loss: 13.32451057434082
Epoch 9, Loss: 13.319207191467285
Epoch 10, Loss: 13.415970802307129
Epoch 11, Loss: 12.081287384033203
Epoch 12, Loss: 14.585814476013184
Epoch 13, Loss: 15.434314727783203
Epoch 14, Loss: 14.828157424926758
Epoch 15, Loss: 11.978022575378418
Epoch 16, Loss: 11.96514892578125
Epoch 17, Loss: 11.233281135559082
Epoch 18, Loss: 12.388664245605469
Epoch 19, Loss: 8.54305648803711
Epoch 20, Loss: 11.612570762634277
Epoch 21, Loss: 10.044716835021973
Epoch 22, Loss: 7.55433464050293
Epoch 23, Loss: 10.714468002319336
Epoch 24, Loss: 9.654027938842773
Epoch 25, Loss: 9.108541488647461
Epoch 26, Loss: 5.98916482925415
Epoch 27, Loss: 7.273525714874268
Epoch 28, Loss: 6.999581813812256
Epoch 29, Loss: 6.052081108093262
Epoch 30,

KeyboardInterrupt: 

In [ ]:

#-------------------------------
"""
    Plot a single IR spectrum.

    Parameters:
    - X: array-like
        The wavenumbers (x-axis values)
    - Y: array-like
        The absorbance or transmittance values (y-axis values)
    - title: str, default='IR Spectrum'
        Title of the plot
    - xlabel: str, default='Wavenumber (cm⁻¹)'
        Label for the x-axis
    - ylabel: str, default='Absorbance'
        Label for the y-axis
    - filename: str, default='ir_spectrum.png'
        Filename to save the plot
"""
"""
def plot_ir(X, Y, title='IR Spectrum', xlabel='Wavenumber (cm⁻¹)', ylabel='Absorbance', filename='ir_spectrum.png'):
    plt.figure(figsize=(10, 6))
    plt.plot(X, Y)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

    print(f"IR spectrum plot saved as '{filename}'")

# Example usage:
plot_ir(processed_X_values, processed_Y_values[2])
"""

In [92]:
def get_indices_above_threshold(vector, threshold):
    return [i for i, value in enumerate(vector) if value > threshold]

# Example usage:
model_output = nn_model(processed_Y_values[2])
threshold = 0.9  # Adjust this threshold as needed
indices_above_threshold = get_indices_above_threshold(model_output, threshold)
correct = get_indices_above_threshold(processed_Labels[2], threshold)

print("Indices Predicted:", indices_above_threshold)
print("Actual Indices:", correct)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat2 in method wrapper_CUDA_mm)

In [93]:
def evaluate_model_accuracy(model, X_values, y_true, threshold=0.9):
    model.eval()
    total_correct = 0
    total_predicted = 0
    total_actual = 0

    with torch.no_grad():
        for i in range(len(X_values)):
            model_output = model(X_values[i].unsqueeze(0)).squeeze()
            predicted_indices = set(get_indices_above_threshold(model_output, threshold))
            actual_indices = set(get_indices_above_threshold(y_true[i], threshold))

            correct = predicted_indices.intersection(actual_indices)
            total_correct += len(correct)
            total_predicted += len(predicted_indices)
            total_actual += len(actual_indices)

    precision = total_correct / total_predicted if total_predicted > 0 else 0
    recall = total_correct / total_actual if total_actual > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")

    return precision, recall, f1_score

# Usage
precision, recall, f1_score = evaluate_model_accuracy(nn_model, processed_Y_values, processed_Labels)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat2 in method wrapper_CUDA_mm)

In [81]:
# Export the model for future training
torch.save({
    'model_state_dict': nn_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item()
}, 'ir_model_checkpoint_v1.pth')

print("Model saved successfully for future training.")


Model saved successfully for future training.
